In [1]:
!pip install unsloth
!pip install rouge-score bert-score datasets accelerate trl peft bitsandbytes
!pip install datasets==3.6.0
import datasets
dataset = datasets.load_dataset('csebuetnlp/xlsum','serbian_latin')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 229.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 169.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 187.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 178.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

xlsum.py: 0.00B [00:00, ?B/s]

serbian_latin/train/0000.parquet:   0%|          | 0.00/27.1M [00:00<?, ?B/s]

serbian_latin/test/0000.parquet:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

serbian_latin/validation/0000.parquet:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7276 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/909 [00:00<?, ? examples/s]

In [2]:

print(f"Train: {len(dataset['train'])}")
print(f"Valid: {len(dataset['validation'])}")
print(f"Test:  {len(dataset['test'])}")

ex = dataset["train"][0]
print("\nTitle:", ex["title"])
print("Article:", ex["text"][:300], "...")
print("Summary:", ex["summary"])


Train: 7276
Valid: 909
Test:  909

Title: Mirijam Menken: Naučnica koja je stvorila život u epruveti
Article: Priča o Mirijam Menkin malo je drugačija. Jednog utorka u februaru 1944. godine, ova 43-godišnja laboratorijska tehničarka ostala je budna čitavu noć umirujući osmomesečnu ćerku - „uzorak uživo", imala je običaj da kaže za nju - kojoj su upravo počeli da rastu zubi. Narednog jutra, Menkin je otišla  ...
Summary: U klasičnoj priči o naučnom otkriću, istraživač radi do duboko u noć, sam u laboratoriji. Odjednom, na pamet mu pada genijalna pomisao: jabuka mu padne na glavu, munje zvekne ključ, pojave se bakterije u Petrijevoj posudi. I eureka: otkriće!


In [3]:
import re

def clean_text(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def filter_by_length(example):
    article_words = len(example["text"].split())
    summary_words = len(example["summary"].split())
    return 50 < article_words < 3000 and 10 < summary_words < 300

dataset_filtered = dataset.filter(filter_by_length)
print(f"После фильтрации — Train: {len(dataset_filtered['train'])}, "
      f"Valid: {len(dataset_filtered['validation'])}, "
      f"Test: {len(dataset_filtered['test'])}")

SYSTEM_PROMPT = (
    "Ti si AI asistent specijalizovan za sažimanje novinskih članaka "
    "na srpskom jeziku. Napravi kratak i informativan sažetak datog članka."
)

def format_example(example):
    """Формирует чат-сообщение для SFT"""
    article = clean_text(example["text"])
    summary = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Sažmi sledeći članak:\n\n{article}"},
        {"role": "assistant", "content": summary},
    ]
    return {"messages": messages}

train_dataset = dataset_filtered["train"].map(format_example)
val_dataset = dataset_filtered["validation"].map(format_example)
test_dataset = dataset_filtered["test"].map(format_example)

print(f"Готово: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")
print("\nПример messages[0]:", train_dataset[0]["messages"][1]["content"][:200])

Filter:   0%|          | 0/7276 [00:00<?, ? examples/s]

Filter:   0%|          | 0/909 [00:00<?, ? examples/s]

Filter:   0%|          | 0/909 [00:00<?, ? examples/s]

После фильтрации — Train: 6602, Valid: 887, Test: 894


Map:   0%|          | 0/6602 [00:00<?, ? examples/s]

Map:   0%|          | 0/887 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Готово: 6602 train, 887 val, 894 test

Пример messages[0]: Sažmi sledeći članak:

Priča o Mirijam Menkin malo je drugačija. Jednog utorka u februaru 1944. godine, ova 43-godišnja laboratorijska tehničarka ostala je budna čitavu noć umirujući osmomesečnu ćerku


In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-14B-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
)

print(model.print_trainable_parameters())


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-14b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


trainable params: 64,225,280 || all params: 14,832,532,480 || trainable%: 0.4330
None


In [5]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen3_14b_serbian_sum",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=50,
    learning_rate=5e-5,
    average_tokens_across_devices=False,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/6602 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=30):   0%|          | 0/6602 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/887 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=30):   0%|          | 0/887 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,717 | Num Epochs = 2 | Total steps = 1,180
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 64,225,280 of 14,832,532,480 (0.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
250,1.809124,1.745733
500,1.739079,1.710788
750,1.704331,1.697532
1000,1.684663,1.692272


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=1180, training_loss=1.751117428278519, metrics={'train_runtime': 4549.5931, 'train_samples_per_second': 2.074, 'train_steps_per_second': 0.259, 'total_flos': 1.5968504687179162e+18, 'train_loss': 1.751117428278519, 'epoch': 2.0})

In [8]:
model.save_pretrained("qwen3_14b_serbian_sum_lora")
tokenizer.save_pretrained("qwen3_14b_serbian_sum_lora")

import shutil
shutil.make_archive("/content/lora_adapter_qwen3_14b", "zip", "/content/qwen3_14b_serbian_sum_lora")

from google.colab import files
files.download("/content/lora_adapter_qwen3_14b.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>